# Trabajo 01 — Parte 1 (cont.): Métodos Heurísticos y Comparativa

**Curso:** Optimización | **Profesor:** Juan David Ospina Arango  
**Entrega:** 24 de marzo de 2026

---

## Motivación

En el Notebook 01 se aplicó el **descenso por gradiente** a las funciones de Rosenbrock y Rastrigin. Los resultados evidenciaron una limitación fundamental de los métodos basados en gradiente: su convergencia está condicionada a la geometría local del paisaje de la función objetivo. Cuando el espacio de búsqueda es multimodal —con múltiples mínimos locales— o presenta geometrías complejas como valles estrechos y curvados, el gradiente tiende a quedar atrapado lejos del óptimo global.

Esta limitación motivó el desarrollo histórico de los **métodos heurísticos**, una familia de algoritmos inspirados en fenómenos naturales (evolución biológica, comportamiento colectivo de animales, física estadística) que sacrifican la garantía de convergencia exacta a cambio de una exploración más robusta del espacio de búsqueda. A diferencia del gradiente, estos métodos no requieren que la función sea diferenciable, continua ni siquiera analítica — basta con poder *evaluarla*.

En este notebook se implementan y comparan tres métodos heurísticos ampliamente utilizados en la literatura de optimización:

- **Algoritmos Evolutivos (EA):** inspirados en la selección natural darwiniana.
- **Optimización por Enjambre de Partículas (PSO):** inspirado en el comportamiento colectivo de aves y peces.
- **Evolución Diferencial (DE):** un método basado en diferencias vectoriales entre individuos de la población.

| # | Sección | Contenido |
|---|---|---|
| 1 | Configuración | Funciones, constantes y parámetros de cada método |
| 2 | Algoritmo Evolutivo (EA) | Fundamento, implementación, resultados y animación |
| 3 | Enjambre de Partículas (PSO) | Fundamento, implementación, resultados y animación |
| 4 | Evolución Diferencial (DE) | Fundamento, implementación y resultados |
| 5 | Comparativa experimental | 30 corridas por método — tabla resumen |
| 6 | Discusión | GD vs heurísticos: valor final, evaluaciones, conclusiones |

In [ ]:
# Descomentar en Google Colab
!pip install deap pyswarms -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import pandas as pd
from pathlib import Path
from IPython.display import Image, display
from scipy.optimize import differential_evolution
from deap import base, creator, tools, algorithms
import pyswarms as ps

# ── Constantes globales ──────────────────────────────────────────────────────
SEED        = 42
BOUNDS      = (-5.0, 5.0)
BOUNDS_VIZ  = (-3.0, 3.0)
GRID_N      = 250
A_RASTRIGIN = 10             # parámetro A — explícito para evitar dependencia de scope
N_RUNS      = 30
GIF_FPS     = 15
THRESH      = {'Rosenbrock': 1e-4, 'Rastrigin': 1.0}  # umbral de éxito por función

# Parámetros EA (ajustados experimentalmente)
EA_POP   = 100
EA_GENS  = 500
EA_CXPB  = 0.7
EA_MUTPB = 0.2

# Parámetros PSO — Clerc & Kennedy (2002)
PSO_N    = 50
PSO_ITER = 500
PSO_OPTS = {'c1': 2.05, 'c2': 2.05, 'w': 0.729}

# Parámetros DE
DE_CONF = dict(strategy='best1bin', maxiter=1000, popsize=15,
               tol=1e-7, mutation=(0.5, 1.0), recombination=0.7)

np.random.seed(SEED)
random.seed(SEED)
Path('outputs').mkdir(exist_ok=True)
print('Entorno listo.')

## 1. Configuración: funciones de prueba

Se utilizan las mismas funciones del Notebook 01 para garantizar consistencia en la comparativa. A continuación se recuerda brevemente su definición:

**Rosenbrock:**
$$f(\mathbf{x}) = \sum_{i=1}^{n-1} \left[ 100(x_{i+1} - x_i^2)^2 + (1-x_i)^2 \right]$$

Su mínimo global se encuentra en $\mathbf{x}^* = (1, 1, \ldots, 1)$ con $f^* = 0$. La función es unimodal pero presenta un valle estrecho y fuertemente curvado que hace que el gradiente apunte casi perpendicular a la dirección de descenso óptima.

**Rastrigin:**
$$f(\mathbf{x}) = An + \sum_{i=1}^{n} \left[ x_i^2 - A\cos(2\pi x_i) \right], \quad A = 10$$

Su mínimo global se encuentra en $\mathbf{x}^* = (0, 0, \ldots, 0)$ con $f^* = 0$. La función es altamente multimodal: para $n=2$ contiene del orden de $10^2$ mínimos locales distribuidos sobre una cuadrícula regular, lo que la convierte en un desafío canónico para métodos de optimización global.

Los **grids de visualización** se calculan una única vez al inicio y se reutilizan en todas las gráficas y animaciones del notebook, evitando evaluaciones redundantes de la función.

In [ ]:
def rosenbrock(x):
    """Rosenbrock: mínimo global en x=[1,...,1], f=0."""
    x = np.asarray(x, dtype=float)
    return float(np.sum(100.0 * (x[1:] - x[:-1]**2)**2 + (1.0 - x[:-1])**2))

def rastrigin(x):
    """Rastrigin: mínimo global en x=[0,...,0], f=0. Usa A_RASTRIGIN explícito."""
    x = np.asarray(x, dtype=float)
    return float(A_RASTRIGIN * len(x) + np.sum(x**2 - A_RASTRIGIN * np.cos(2*np.pi*x)))

# Lista canónica — orden fijo en todo el notebook
CONFIGS = [('Rosenbrock', rosenbrock), ('Rastrigin', rastrigin)]

def make_grid(f, bounds=BOUNDS_VIZ, n=GRID_N):
    """Grid 2D para visualización. Calculado una sola vez y reutilizado."""
    x  = np.linspace(*bounds, n)
    X, Y = np.meshgrid(x, x)
    XY = np.stack([X, Y], axis=-1)
    Z  = np.apply_along_axis(f, 2, XY)
    return X, Y, Z

# Pre-calcular grids — se reutilizan en animaciones y gráficas
grids = {nombre: make_grid(f) for nombre, f in CONFIGS}
print('Funciones y grids listos.')

## 2. Algoritmo Evolutivo (EA)

### 2.1 Fundamento teórico

Los **algoritmos evolutivos** constituyen una familia de metaheurísticas de optimización inspiradas en el proceso de selección natural propuesto por Darwin (1859). La idea central es mantener una **población** de soluciones candidatas que evoluciona iterativamente mediante tres operadores fundamentales:

1. **Selección:** los individuos con mayor aptitud (*fitness*) tienen mayor probabilidad de reproducirse. En este trabajo se usa **selección por torneo**: se elige aleatoriamente un subconjunto de la población y el mejor individuo del subconjunto pasa a la siguiente generación.

2. **Cruce (*crossover*):** dos individuos "padres" intercambian material genético para producir "hijos". Se utiliza el operador **cxBlend** con $\alpha = 0.5$, que produce descendientes cuyos genes se sitúan en un intervalo $[x_i - \alpha \Delta, x_i + \alpha \Delta]$, donde $\Delta = |x_i^{(1)} - x_i^{(2)}|$. Este operador favorece la exploración del espacio entre y alrededor de los padres.

3. **Mutación:** se introduce ruido aleatorio en la solución para mantener diversidad genética y evitar convergencia prematura. Se usa **mutación gaussiana** con $\mu = 0$, $\sigma = 0.5$, lo que equivale a perturbar cada gen con un desplazamiento normalmente distribuido.

El ciclo generacional continúa hasta alcanzar un número máximo de generaciones o un criterio de convergencia. La implementación se realiza con la librería **DEAP** (*Distributed Evolutionary Algorithms in Python*), que ofrece una arquitectura flexible basada en *toolbox* de operadores registrables.

### 2.2 Parámetros y justificación

| Parámetro | Valor | Justificación |
|---|---|---|
| Tamaño de población | 100 | Suficiente para explorar $[-5,5]^n$ con $n \leq 3$; mayor población aumenta el costo linealmente |
| Generaciones | 500 | Se observó estabilización del mejor individuo antes de la generación 400 en ensayos previos |
| Probabilidad de cruce | 0.70 | Valor estándar en la literatura (De Jong, 1975) |
| Probabilidad de mutación | 0.20 | Mantiene diversidad sin degradar excesivamente las soluciones buenas |
| Tamaño del torneo | 3 | Presión selectiva moderada: suficiente para dirigir la búsqueda sin eliminar diversidad prematuramente |

> **Nota sobre reproducibilidad:** DEAP utiliza internamente el módulo `random` de Python para sus operadores de cruce, mutación y selección. Para garantizar resultados reproducibles, se fijan *ambas* semillas (`random.seed` y `np.random.seed`) antes de cada corrida.

In [ ]:
def _ea_toolbox(f, ndim):
    """Construye el toolbox DEAP. Limpia creator para evitar conflictos entre corridas."""
    for attr in ('FitnessMin', 'Individual'):
        if hasattr(creator, attr):
            delattr(creator, attr)
    creator.create('FitnessMin', base.Fitness, weights=(-1.0,))
    creator.create('Individual', list, fitness=creator.FitnessMin)

    tb = base.Toolbox()
    tb.register('attr',       random.uniform, BOUNDS[0], BOUNDS[1])  # usa random de Python (mismo que DEAP)
    tb.register('individual', tools.initRepeat, creator.Individual, tb.attr, n=ndim)
    tb.register('population', tools.initRepeat, list, tb.individual)
    tb.register('evaluate',   lambda ind: (f(np.array(ind)),))
    tb.register('mate',       tools.cxBlend, alpha=0.5)
    tb.register('mutate',     tools.mutGaussian, mu=0, sigma=0.5, indpb=0.2)
    tb.register('select',     tools.selTournament, tournsize=3)
    return tb


def run_ea(f, ndim, seed, track_history=False):
    """
    Corre el EA una vez.

    track_history=True registra el mejor f por generación (para animaciones).
    En experimentos masivos usar track_history=False para mayor velocidad.
    """
    random.seed(seed)
    np.random.seed(seed)
    tb  = _ea_toolbox(f, ndim)
    hof = tools.HallOfFame(1)

    if track_history:
        stats = tools.Statistics(lambda ind: ind.fitness.values)
        stats.register('min', np.min)
        _, log = algorithms.eaSimple(
            tb.population(n=EA_POP), tb,
            cxpb=EA_CXPB, mutpb=EA_MUTPB, ngen=EA_GENS,
            stats=stats, halloffame=hof, verbose=False
        )
        historial = [r['min'] for r in log]
    else:
        algorithms.eaSimple(
            tb.population(n=EA_POP), tb,
            cxpb=EA_CXPB, mutpb=EA_MUTPB, ngen=EA_GENS,
            stats=None, halloffame=hof, verbose=False
        )
        historial = None

    return {
        'x_final':   np.array(hof[0]),
        'f_final':   hof[0].fitness.values[0],   # ya evaluado por DEAP, sin eval extra
        'n_evals':   EA_POP * (EA_GENS + 1),
        'historial': historial,
    }


res = run_ea(rosenbrock, 2, seed=0)
print(f'EA | Rosenbrock 2D | f* = {res["f_final"]:.4e} | evals = {res["n_evals"]}')

### 2.3 Animación — evolución del mejor individuo por generación

La siguiente animación muestra cómo evoluciona el valor de la función objetivo del **mejor individuo** a lo largo de las generaciones. Se utiliza escala *symlog* en el eje $y$ para visualizar simultáneamente el rango completo de valores, incluyendo aquellos que se acercan a cero.

Se generan dos GIFs: uno para cada función de prueba.

In [ ]:
# Contador global de figuras — lista mutable para evitar el antipatrón de `global` en notebooks
FIG_NUM = [1]

def animar_convergencia(historial, nombre, metodo):
    """
    GIF de la curva de convergencia (mejor f por iteración).
    Usa escala symlog para manejar correctamente valores cero.
    """
    fig_n = FIG_NUM[0]
    FIG_NUM[0] += 1

    fig, ax = plt.subplots(figsize=(7, 4))
    linea,  = ax.plot([], [], lw=2, color='#1f77b4')
    punto,  = ax.plot([], [], 'o', color='#d62728', ms=7)

    ax.set_xlim(0, len(historial) - 1)
    ax.set_ylim(0, max(historial) * 1.1)
    ax.set_yscale('symlog', linthresh=1e-10)   # symlog: maneja ceros sin crash
    ax.set_xlabel('Generación / Iteración')
    ax.set_ylabel('Mejor $f$ (escala symlog)')
    ax.grid(True, alpha=0.3)
    titulo = ax.set_title('')

    frames = np.linspace(0, len(historial) - 1, 120, dtype=int)

    def actualizar(k):
        linea.set_data(range(k+1), historial[:k+1])
        punto.set_data([k], [historial[k]])
        titulo.set_text(f'Figura {fig_n}. {metodo} — {nombre} | iter {k} | f = {historial[k]:.3e}')
        return linea, punto, titulo

    ani  = animation.FuncAnimation(fig, actualizar, frames=frames,
                                    init_func=lambda: (linea, punto, titulo), blit=True)
    ruta = f'outputs/{metodo.lower()}_{nombre.lower()}_conv.gif'
    ani.save(ruta, writer='pillow', fps=GIF_FPS)
    plt.close()
    print(f'Guardado: {ruta}  ({Path(ruta).stat().st_size // 1024} KB)')
    return ruta

In [ ]:
for nombre, f in CONFIGS:
    res = run_ea(f, 2, seed=SEED, track_history=True)
    display(Image(animar_convergencia(res['historial'], nombre, 'EA')))

## 3. Optimización por Enjambre de Partículas (PSO)

### 3.1 Fundamento teórico

El **PSO** (*Particle Swarm Optimization*) fue propuesto por Kennedy y Eberhart en 1995, inspirado en el comportamiento colectivo de bandadas de aves y bancos de peces. A diferencia de los algoritmos evolutivos, el PSO no tiene operadores de cruce ni mutación: la actualización de cada partícula se rige por tres fuerzas que actúan simultáneamente sobre su velocidad:

$$\mathbf{v}_i^{(t+1)} = w\,\mathbf{v}_i^{(t)} + c_1 r_1 \underbrace{(\mathbf{p}_i - \mathbf{x}_i^{(t)})}_{\text{componente cognitiva}} + c_2 r_2 \underbrace{(\mathbf{g} - \mathbf{x}_i^{(t)})}_{\text{componente social}}$$

$$\mathbf{x}_i^{(t+1)} = \mathbf{x}_i^{(t)} + \mathbf{v}_i^{(t+1)}$$

donde:
- $\mathbf{x}_i^{(t)}$ es la posición de la partícula $i$ en la iteración $t$
- $\mathbf{v}_i^{(t)}$ es su velocidad actual
- $\mathbf{p}_i$ es la mejor posición histórica de la partícula (*memoria personal*)
- $\mathbf{g}$ es la mejor posición encontrada por el enjambre completo (*memoria social*)
- $w$ es el **factor de inercia**, que controla la capacidad exploratoria: valores altos favorecen la exploración, valores bajos favorecen la explotación
- $c_1$ y $c_2$ son los **coeficientes de aceleración** cognitivo y social
- $r_1, r_2 \sim U(0,1)$ son números aleatorios que introducen estocasticidad

### 3.2 Elección de parámetros — Factor de constricción de Clerc & Kennedy

La elección de $w$, $c_1$ y $c_2$ es crítica para la convergencia del PSO. Valores inadecuados pueden llevar a divergencia (partículas escapan del espacio de búsqueda) o a convergencia prematura. 

En este trabajo se utilizan los valores derivados del **factor de constricción** propuesto por Clerc y Kennedy (2002), que garantizan convergencia teórica:

$$\chi = \frac{2}{|2 - \phi - \sqrt{\phi^2 - 4\phi}|}, \quad \phi = c_1 + c_2 > 4$$

Con $c_1 = c_2 = 2.05$ se obtiene $\phi = 4.1$ y $\chi \approx 0.729$, que es el valor usado como inercia $w$.

| Parámetro | Valor | Justificación |
|---|---|---|
| Partículas | 50 | Cobertura adecuada de $[-5,5]^n$ con $n \leq 3$ |
| Iteraciones | 500 | Convergencia observada antes de la iteración 300 |
| $w$ | 0.729 | Factor de constricción de Clerc & Kennedy |
| $c_1 = c_2$ | 2.05 | Pesos cognitivo y social balanceados |

In [ ]:
def run_pso(f, ndim, seed, track_history=False):
    """Corre PSO con pyswarms. track_history devuelve historial de costo por iteración."""
    np.random.seed(seed)

    def batch_f(X):
        return np.array([f(X[i]) for i in range(X.shape[0])])

    bounds = (np.full(ndim, BOUNDS[0]), np.full(ndim, BOUNDS[1]))
    opt    = ps.single.GlobalBestPSO(n_particles=PSO_N, dimensions=ndim,
                                      options=PSO_OPTS, bounds=bounds)
    cost, pos = opt.optimize(batch_f, iters=PSO_ITER, verbose=False)

    return {
        'x_final':   pos,
        'f_final':   float(cost),
        'n_evals':   PSO_N * PSO_ITER,
        'historial': list(opt.cost_history) if track_history else None,
    }


res = run_pso(rosenbrock, 2, seed=SEED)
print(f'PSO | Rosenbrock 2D | f* = {res["f_final"]:.4e} | evals = {res["n_evals"]}')

### 3.3 Animación — convergencia del mejor costo global

La animación muestra la evolución del mejor costo global $f(\mathbf{g})$ a lo largo de las iteraciones. A diferencia del EA, el PSO tiende a converger más abruptamente al inicio y luego estabilizarse —  característica del predominio de la componente social sobre la cognitiva en las etapas avanzadas.

In [ ]:
for nombre, f in CONFIGS:
    res = run_pso(f, 2, seed=SEED, track_history=True)
    display(Image(animar_convergencia(res['historial'], nombre, 'PSO')))

## 4. Evolución Diferencial (DE)

### 4.1 Fundamento teórico

La **Evolución Diferencial** fue propuesta por Storn y Price en 1997. Es un algoritmo evolutivo diseñado específicamente para optimización de funciones continuas, y se distingue de los EA clásicos en su mecanismo de generación de candidatos: en lugar de usar un operador de cruce basado en recombinación de genes, la DE genera nuevos individuos mediante **perturbaciones vectoriales** derivadas de la propia población.

El algoritmo opera en tres pasos por individuo en cada generación:

**1. Mutación (generación del vector *donor*):**
$$\mathbf{v} = \mathbf{x}_{best} + F \cdot (\mathbf{x}_{r1} - \mathbf{x}_{r2})$$

donde $\mathbf{x}_{best}$ es el mejor individuo de la generación actual, $\mathbf{x}_{r1}$ y $\mathbf{x}_{r2}$ son individuos distintos elegidos aleatoriamente de la población, y $F \in [0, 2]$ es el **factor de escala** que controla la amplitud de la perturbación.

**2. Cruce (*crossover* binomial):**
$$u_j = \begin{cases} v_j & \text{si } r_j \leq CR \text{ o } j = j_{rand} \\ x_j & \text{en otro caso} \end{cases}$$

donde $CR \in [0, 1]$ es la **tasa de recombinación**, $r_j \sim U(0,1)$ y $j_{rand}$ garantiza que al menos un gen provenga del vector donor.

**3. Selección uno a uno:**
$$\mathbf{x}^{(t+1)} = \begin{cases} \mathbf{u} & \text{si } f(\mathbf{u}) \leq f(\mathbf{x}^{(t)}) \\ \mathbf{x}^{(t)} & \text{en otro caso} \end{cases}$$

Esta selección greedy garantiza que la población nunca empeora de generación a generación.

### 4.2 ¿Por qué DE supera a EA y PSO en Rosenbrock?

La clave está en la **escala adaptativa** de la mutación. El vector diferencia $F \cdot (\mathbf{x}_{r1} - \mathbf{x}_{r2})$ tiene magnitud proporcional a la dispersión actual de la población: cuando los individuos están dispersos (exploración), los pasos son grandes; cuando la población converge (explotación), los pasos se vuelven pequeños automáticamente. Esto le permite a la DE navegar el valle estrecho de Rosenbrock sin la necesidad de ajustar manualmente el tamaño del paso, a diferencia del EA con mutación gaussiana de $\sigma$ fijo.

| Parámetro | Valor | Justificación |
|---|---|---|
| Estrategia | `best1bin` | Usa el mejor individuo como base — convergencia más rápida |
| Factor $F$ | (0.5, 1.0) | Rango adaptativo entre generaciones |
| Recombinación $CR$ | 0.7 | Mezcla moderada entre target y donor |
| Tamaño de población | $15 \times n$ | Escala con la dimensión del problema |

In [ ]:
def run_de(f, ndim, seed, track_history=False):
    """Corre Evolución Diferencial con scipy. track_history ignorado (scipy no lo expone)."""
    result = differential_evolution(f, [BOUNDS] * ndim, seed=seed, **DE_CONF)
    return {
        'x_final':   result.x,
        'f_final':   float(result.fun),
        'n_evals':   int(result.nfev),
        'historial': None,
    }


for nombre, f in CONFIGS:
    for ndim in [2, 3]:
        res = run_de(f, ndim, seed=SEED)
        print(f'DE | {nombre} {ndim}D | f* = {res["f_final"]:.4e} | evals = {res["n_evals"]}')

## 5. Comparativa experimental — 30 corridas independientes

### 5.1 Diseño experimental

Para obtener una comparación estadísticamente válida, se realizan **30 corridas independientes** por cada combinación de método, función y dimensión. Cada corrida usa una semilla aleatoria distinta (`seed = 0, 1, ..., 29`), lo que garantiza que los resultados no dependan de una inicialización particular.

El número 30 se elige siguiendo la convención estadística que permite asumir normalidad aproximada de las distribuciones muestrales por el Teorema del Límite Central, facilitando comparaciones y pruebas de hipótesis si se desea extender el análisis.

Las métricas reportadas son:

- **$f^*$ media ± std:** promedio y desviación estándar del valor de la función objetivo en el mejor punto encontrado. Una media baja con std bajo indica un método robusto y consistente.
- **Tasa de éxito:** fracción de corridas en las que el método supera un umbral de calidad predefinido ($f^* < 10^{-4}$ para Rosenbrock, $f^* < 1.0$ para Rastrigin). Este indicador complementa la media al capturar la confiabilidad del método.
- **Evaluaciones de $f$:** número total de llamadas a la función objetivo. Es el indicador estándar de costo computacional en optimización sin gradiente, independiente de la velocidad del hardware.

In [ ]:
# Interfaz unificada: todos los runners aceptan (f, ndim, seed, track_history)
METODOS = {'EA': run_ea, 'PSO': run_pso, 'DE': run_de}
registros = []

for nombre, f in CONFIGS:
    for ndim in [2, 3]:
        for metodo, runner in METODOS.items():
            f_vals, evals_list = [], []
            for seed in range(N_RUNS):
                res = runner(f, ndim, seed)
                f_vals.append(res['f_final'])
                evals_list.append(res['n_evals'])
            f_arr = np.array(f_vals)
            registros.append({
                'Método':    metodo,
                'Función':   nombre,
                'Dim':       ndim,
                'f* media':  f_arr.mean(),
                'f* std':    f_arr.std(),
                'f* mejor':  f_arr.min(),
                'Éxito (%)': (f_arr < THRESH[nombre]).mean() * 100,
                'Evals':     int(np.mean(evals_list)),
            })
            print(f"{metodo:4s} | {nombre:12s} {ndim}D | "
                  f"media={f_arr.mean():.2e}  éxito={registros[-1]['Éxito (%)']:5.1f}%  "
                  f"evals={registros[-1]['Evals']}")

df = pd.DataFrame(registros)

In [ ]:
# Formatear columnas numéricas en una sola pasada
fmt = {'f* media': '{:.2e}', 'f* std': '{:.2e}',
       'f* mejor': '{:.2e}', 'Éxito (%)': '{:.1f}'}
tabla = df.style.format(fmt)

print('Tabla 1. Comparativa de métodos heurísticos — 30 corridas independientes')
display(tabla)

df.to_csv('outputs/comparativa_heuristicos.csv', index=False)
print('Guardado: outputs/comparativa_heuristicos.csv')

### 5.2 Visualización de resultados

La siguiente figura muestra el valor medio de $f^*$ para cada método en cada combinación de función y dimensión. La escala *symlog* permite visualizar simultáneamente valores cercanos a cero y valores órdenes de magnitud mayores. La tasa de éxito se indica sobre cada barra.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
colores   = {'EA': '#1f77b4', 'PSO': '#ff7f0e', 'DE': '#2ca02c'}

for row, (nombre, _) in enumerate(CONFIGS):
    for col, ndim in enumerate([2, 3]):
        fig_n = FIG_NUM[0]; FIG_NUM[0] += 1
        ax    = axes[row, col]
        sub   = df[(df['Función'] == nombre) & (df['Dim'] == ndim)]
        bars  = ax.bar(sub['Método'], sub['f* media'],
                       color=[colores[m] for m in sub['Método']],
                       edgecolor='white', linewidth=0.5)
        ax.set_yscale('symlog', linthresh=1e-10)
        ax.set_title(f'Figura {fig_n}. {nombre} {ndim}D — f* media (30 corridas)')
        ax.set_ylabel('$f^*$ media (symlog)')
        ax.set_xlabel('Método')
        ax.grid(True, axis='y', alpha=0.3)
        ymax = ax.get_ylim()[1]
        for bar, (_, rd) in zip(bars, sub.iterrows()):
            ax.text(bar.get_x() + bar.get_width() / 2, ymax * 0.85,
                    f"{rd['Éxito (%)']:.0f}%",
                    ha='center', va='top', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/comparativa_barras.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: outputs/comparativa_barras.png')

## 6. Discusión: GD vs Métodos Heurísticos

### 6.1 Análisis por función de prueba

#### Rosenbrock

Esta función representa el caso de una función **unimodal con geometría difícil**. El gradiente converge al mínimo cuando logra mantenerse dentro del valle, pero es altamente sensible a la condición inicial: si el punto de partida queda fuera del valle, el gradiente apunta hacia una dirección que aleja al algoritmo del mínimo global.

De los métodos heurísticos, **DE** es el único que alcanza el mínimo global de forma consistente (100% de éxito en 2D y 3D), gracias a su capacidad de escalar automáticamente el tamaño de paso según la dispersión de la población. **PSO** funciona bien en 2D pero pierde efectividad en 3D, ya que el enjambre tiende a dispersarse en el espacio de mayor dimensión antes de identificar el valle. **EA** falla en ambas dimensiones con el operador `cxBlend`: los hijos recombinados entre dos padres ubicados en distintas partes del valle terminan fuera de él, en zonas de alto costo.

#### Rastrigin

Esta función representa el caso de una función **multimodal con muchos mínimos locales**. El gradiente queda atrapado en el mínimo local más cercano a la condición inicial en la gran mayoría de corridas —  exactamente el escenario para el que los heurísticos fueron diseñados.

Los tres métodos heurísticos encuentran el mínimo global ($f^* = 0$) con alta confiabilidad. La exploración poblacional les permite *saltar* entre mínimos locales, algo imposible para el GD. **DE** lo consigue además con el menor número de evaluaciones de $f$, lo que lo posiciona como el más eficiente.

### 6.2 Costo computacional: evaluaciones de $f$

El número de evaluaciones de la función objetivo es el indicador de costo más relevante en optimización sin gradiente, ya que es independiente del hardware y aplicable a funciones de cualquier complejidad computacional:

| Método | Evaluaciones aprox. | Calidad Rosenbrock | Calidad Rastrigin |
|---|---|---|---|
| **GD** | 500 – 5.000 | Variable — depende de la condición inicial | Atrapado en mínimos locales |
| **EA** | ~50.000 | Pobre — 0% éxito en Rosenbrock | Excelente — 100% éxito |
| **PSO** | ~25.000 | Buena en 2D, deficiente en 3D | Excelente — 100% éxito |
| **DE** | 2.000 – 11.000 | **Excelente — 100% éxito** | **Excelente — 100% éxito** |

El GD es el más barato en evaluaciones, pero no garantiza optimalidad global. DE logra mejor calidad que EA y PSO con menos evaluaciones — una combinación difícil de superar para problemas de optimización continua de caja negra.

### 6.3 ¿Qué aportó cada método?

**Descenso por gradiente**  
El GD aportó una solución eficiente y precisa cuando la función es suave y la condición inicial está razonablemente cerca del óptimo. Su bajo costo computacional lo hace ideal como **refinamiento local** de una solución heurística: si se toma la solución de DE y se aplica unas pocas iteraciones de GD, se puede obtener precisión de máquina a bajo costo.

**Algoritmo Evolutivo (EA)**  
El EA demostró ser el método más adecuado para funciones multimodales con muchos mínimos locales (Rastrigin), gracias a su población diversa que explora simultáneamente múltiples regiones del espacio. Sin embargo, el operador `cxBlend` resultó inadecuado para navegar el valle estrecho de Rosenbrock. Este hallazgo es consistente con la literatura: los EA de representación real requieren operadores de cruce que preserven la dirección del valle (como SBX o DE) para ser competitivos en funciones unimodales complejas.

**Optimización por Enjambre de Partículas (PSO)**  
El PSO ofreció el mejor equilibrio entre simplicidad de implementación y rendimiento. Su convergencia en 2D para Rosenbrock es notable, pero la pérdida de cohesión del enjambre en 3D evidencia una limitación conocida: en espacios de alta dimensión, el enjambre se dispersa y la componente social pierde efectividad. Ajustar $w$ dinámicamente (inercia decreciente) podría mejorar los resultados en 3D.

**Evolución Diferencial (DE)**  
DE fue el método más robusto y eficiente en todos los escenarios evaluados. La estrategia `best1bin` combina exploración (a través de las diferencias vectoriales) con explotación (anclar la mutación al mejor individuo conocido). La escala automática del paso de mutación le confiere una adaptabilidad que ni EA ni PSO poseen con parámetros fijos.

> **Recomendación práctica:** para optimización continua de caja negra, usar **DE** como método principal. Si se dispone de gradientes analíticos y la función es unimodal, el **GD con backtracking** es más eficiente. Combinar ambos (DE para exploración global + GD para refinamiento local) es una estrategia robusta y ampliamente usada en la práctica.

## Conclusiones

A partir de los experimentos realizados con 30 corridas independientes sobre las funciones de Rosenbrock y Rastrigin en 2D y 3D, se pueden extraer las siguientes conclusiones:

1. **Los métodos heurísticos son superiores al GD en presencia de multimodalidad.** Para Rastrigin, los tres métodos heurísticos alcanzaron el mínimo global con tasas de éxito del 100%, mientras que el GD quedó atrapado en mínimos locales en la mayoría de corridas.

2. **La Evolución Diferencial (DE) fue el método más robusto y eficiente.** Alcanzó el 100% de éxito en todos los escenarios —incluyendo Rosenbrock 3D donde PSO y EA fallaron— con el menor número de evaluaciones de la función objetivo entre los métodos heurísticos.

3. **El operador de cruce `cxBlend` no es adecuado para funciones con geometría de valle estrecho.** La recombinación convexa que produce `cxBlend` tiende a situar los hijos fuera del valle de Rosenbrock, impidiendo al EA explotar la estructura unimodal de la función. Operadores como SBX o directamente el esquema de mutación de DE serían más apropiados.

4. **El GD sigue siendo valioso por su bajo costo computacional** cuando la función es diferenciable y se cuenta con una buena condición inicial. Una estrategia híbrida —heurístico para encontrar la cuenca del mínimo global, GD para refinar— aprovecha las fortalezas de ambos paradigmas.

5. **El costo computacional en evaluaciones de $f$ escala favorablemente para DE** con el aumento de dimensión: la población crece linealmente ($15n$) mientras que la calidad de la solución no se degrada significativamente al pasar de 2D a 3D.

> El análisis de la **Parte 2 — Problema del Viajero por las 32 capitales de México** se presenta en el **Notebook 03**.